# Logistic Regression: Quantization and Noise Robustness

Run this notebook from top to bottom to inspect each evaluation stage without repeatedly running the deployment script. It compares:

- **Baseline:** the trained PyTorch `float32` linear model.
- **FP32 edge-style reference:** the same learned parameters evaluated with NumPy matrix multiplication. This is a software reference, **not** a measurement on MAX78002 hardware.
- **Quantized model:** in-memory INT8 inference using the project's quantizer; this notebook does not export a model file.

The notebook reports clean performance and performance at 12, 6, and 0 dB SNR, including per-class scores and confusion matrices. Its data loader preprocesses ECGs before noise is added, so this experiment evaluates **post-preprocessing noise**; it cannot establish whether preprocessing itself removes noise.

## Stage 1 — Setup

The default data path comes from `ml-model/data_loader.py`. Change `DATA_DIR` below if the MIT-BIH files live elsewhere. The data loader uses SciPy for preprocessing, so its compiled dependencies must be allowed by the local Python/Windows environment.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "ml-model" / "data_loader.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find the repository root. Open this notebook from inside "
        "the ECG-heartbeat-classification repository."
    )

MODEL_DIR = ROOT / "ml-model"
sys.path.insert(0, str(MODEL_DIR))
import data_loader
from int8_export import quantize_linear_parameters, predict_int8_linear

# Set this to a Path/string if your MIT-BIH database is not at the loader's default.
DATA_DIR = data_loader.DATA_DIR
data_loader.DATA_DIR = Path(DATA_DIR)
if not data_loader.DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"MIT-BIH data directory not found: {data_loader.DATA_DIR}. "
        "Set DATA_DIR to the directory containing the record files."
    )

SEED = 42
NUM_CLASSES = 4
CLASS_NAMES = ["N - Normal", "S - Supraventricular", "V - Ventricular", "F - Fusion"]
SNR_LEVELS_DB = (12, 6, 0)
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Repository: {ROOT}")
print(f"MIT-BIH data: {data_loader.DATA_DIR}")

## Stage 2 — Load patient-independent splits

The existing loader provides training and validation records from DS1 and the held-out test records from DS2. Keep the test split for final evaluation only.

In [ ]:
X_train, y_train = data_loader.load_training_data()
X_val, y_val = data_loader.load_validation_data()
X_test, y_test = data_loader.load_testing_data()

for split_name, signals, labels in (
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test),
):
    if signals.ndim != 2 or signals.shape[1] != 256:
        raise ValueError(f"{split_name} signals must have shape (beats, 256); got {signals.shape}.")
    if len(signals) != len(labels) or len(labels) == 0:
        raise ValueError(f"{split_name} signals and labels must be non-empty and aligned.")
    print(f"{split_name}: X={signals.shape}, y={labels.shape}, "
          f"class counts={np.bincount(labels, minlength=NUM_CLASSES).tolist()}")

## Stage 3 — Train the FP32 baseline

This mirrors the logistic-regression training setup in `ml-model/logistic_reg.py`: one `Linear(256, 4)` layer, SGD, cross-entropy, and inverse-frequency weighted sampling. Run this cell once per fresh kernel; later evaluation cells reuse `model`.

In [ ]:
BATCH_SIZE = 64
EPOCHS = 50

train_signals = torch.as_tensor(X_train, dtype=torch.float32)
train_labels = torch.as_tensor(y_train, dtype=torch.long)
val_signals = torch.as_tensor(X_val, dtype=torch.float32)
val_labels = torch.as_tensor(y_val, dtype=torch.long)
train_dataset = TensorDataset(train_signals, train_labels)
class_counts = torch.bincount(train_labels, minlength=NUM_CLASSES)
if torch.any(class_counts == 0):
    raise ValueError("Every class must be present in the training split for weighted sampling.")
sample_weights = (1.0 / class_counts.float())[train_labels]
sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(train_dataset),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(TensorDataset(val_signals, val_labels), batch_size=BATCH_SIZE)

model = nn.Linear(256, NUM_CLASSES)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
training_loss = []
validation_accuracy = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch_signals, batch_labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_signals), batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_labels)
    training_loss.append(total_loss / len(train_dataset))

    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_signals, batch_labels in val_loader:
            correct += (model(batch_signals).argmax(dim=1) == batch_labels).sum().item()
    validation_accuracy.append(correct / len(val_labels))
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:02d}/{EPOCHS}: loss={training_loss[-1]:.4f}, "
              f"validation accuracy={validation_accuracy[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(training_loss)
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].plot(validation_accuracy)
axes[1].set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy")
fig.tight_layout()
plt.show()
model.eval()

## Stage 4 — Define shared metrics and inference paths

Macro F1 is the unweighted mean of the four class F1 scores. Undefined per-class precision/recall/F1 is reported as zero, matching the script's earlier reporting convention.

In [ ]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    if y_true.size == 0 or y_true.shape != y_pred.shape:
        raise ValueError("True labels and predictions must be non-empty and have the same shape.")
    if (np.any((y_true < 0) | (y_true >= NUM_CLASSES)) or
            np.any((y_pred < 0) | (y_pred >= NUM_CLASSES))):
        raise ValueError(f"Labels and predictions must be in [0, {NUM_CLASSES - 1}].")

    confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    np.add.at(confusion, (y_true, y_pred), 1)
    true_positive = np.diag(confusion)
    actual = confusion.sum(axis=1)
    predicted = confusion.sum(axis=0)
    precision = np.divide(true_positive, predicted, out=np.zeros(NUM_CLASSES), where=predicted != 0)
    recall = np.divide(true_positive, actual, out=np.zeros(NUM_CLASSES), where=actual != 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros(NUM_CLASSES), where=(precision + recall) != 0)
    return {
        "accuracy": float(true_positive.sum() / y_true.size),
        "macro_f1": float(f1.mean()),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": confusion,
    }


def predict_baseline(signals):
    with torch.no_grad():
        return model(torch.as_tensor(signals, dtype=torch.float32)).argmax(dim=1).cpu().numpy()


def predict_fp32_edge_reference(signals):
    weights = model.weight.detach().cpu().numpy()
    bias = model.bias.detach().cpu().numpy()
    logits = np.asarray(signals, dtype=np.float32) @ weights.T + bias
    return logits.argmax(axis=1)


quantized_model = quantize_linear_parameters(
    model.weight.detach().cpu().numpy(),
    model.bias.detach().cpu().numpy(),
    input_range=(-3.0, 3.0),
)


def predict_int8(signals):
    logits = predict_int8_linear(np.asarray(signals, dtype=np.float32), quantized_model)
    return logits.argmax(axis=1)


PREDICTORS = {
    "Baseline FP32": predict_baseline,
    "FP32 edge reference": predict_fp32_edge_reference,
    "Quantized INT8": predict_int8,
}
test_signals = np.asarray(X_test, dtype=np.float32)
test_labels = np.asarray(y_test, dtype=np.int64)
fp32_baseline_predictions = predict_baseline(test_signals)
fp32_reference_predictions = predict_fp32_edge_reference(test_signals)
assert np.array_equal(fp32_baseline_predictions, fp32_reference_predictions), (
    "The PyTorch and NumPy FP32 paths should produce matching predictions."
)
print("FP32 PyTorch and NumPy reference predictions match on the clean test set.")

## Stage 5 — Create paired noise conditions

Each beat receives Gaussian noise scaled to its own signal power. A fixed random realization is shared across the SNR levels and across all model paths for a fair comparison. Noise is added to the already filtered, segmented, normalized, and clipped test beats.

In [ ]:
noise_rng = np.random.default_rng(SEED)
standard_noise = noise_rng.normal(size=test_signals.shape).astype(np.float32)
signal_power = np.mean(np.square(test_signals), axis=1, keepdims=True)
noise_power = np.mean(np.square(standard_noise), axis=1, keepdims=True)
if np.any(signal_power <= 0) or np.any(noise_power <= 0):
    raise ValueError("Cannot form SNR-scaled noise for a zero-power test beat.")

TEST_CONDITIONS = {"Clean": test_signals}
for snr_db in SNR_LEVELS_DB:
    target_noise_power = signal_power / (10 ** (snr_db / 10))
    scaled_noise = standard_noise * np.sqrt(target_noise_power / noise_power)
    measured_snr = 10 * np.log10(
        signal_power / np.mean(np.square(scaled_noise), axis=1, keepdims=True)
    )
    if not np.allclose(measured_snr, snr_db, atol=1e-4):
        raise RuntimeError(f"Generated noise does not meet requested {snr_db} dB SNR.")
    TEST_CONDITIONS[f"{snr_db} dB"] = test_signals + scaled_noise

print("Evaluation conditions:", ", ".join(TEST_CONDITIONS))

## Stage 6 — Compare accuracy and macro F1

Run the three inference paths on the same samples at each condition. This table is the high-level robustness summary.

In [ ]:
RESULTS = {}
for condition, signals in TEST_CONDITIONS.items():
    RESULTS[condition] = {}
    for model_name, predict in PREDICTORS.items():
        predictions = predict(signals)
        RESULTS[condition][model_name] = {
            "predictions": predictions,
            **calculate_metrics(test_labels, predictions),
        }

model_names = list(PREDICTORS)
print(f"{'Condition':<12}" + "".join(f"{name + ' Acc':>23}{name + ' F1':>12}" for name in model_names))
for condition, model_results in RESULTS.items():
    print(f"{condition:<12}" + "".join(
        f"{model_results[name]['accuracy']:>23.4f}{model_results[name]['macro_f1']:>12.4f}"
        for name in model_names
    ))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
conditions = list(TEST_CONDITIONS)
for model_name in model_names:
    axes[0].plot(conditions, [RESULTS[c][model_name]["accuracy"] for c in conditions], marker="o", label=model_name)
    axes[1].plot(conditions, [RESULTS[c][model_name]["macro_f1"] for c in conditions], marker="o", label=model_name)
axes[0].set(title="Accuracy vs. noise", ylabel="Accuracy", xlabel="Condition")
axes[1].set(title="Macro F1 vs. noise", ylabel="Macro F1", xlabel="Condition")
for axis in axes:
    axis.grid(True, alpha=0.3)
    axis.legend()
fig.tight_layout()
plt.show()

## Stage 7 — Find the classes most affected

Compare each class's recall/sensitivity and F1 with its clean value. Larger drops indicate greater impact. Confusion matrices show which other classes those beats are assigned to.

In [ ]:
clean_condition = "Clean"
for model_name in model_names:
    print(f"\n{model_name}: per-class recall and F1")
    print(f"{'Condition':<12}{'Class':<25}{'Recall':>10}{'F1':>10}{'Recall Δ':>12}{'F1 Δ':>10}")
    clean_metrics = RESULTS[clean_condition][model_name]
    for condition in TEST_CONDITIONS:
        metrics = RESULTS[condition][model_name]
        for class_index, class_name in enumerate(CLASS_NAMES):
            recall_delta = metrics["recall"][class_index] - clean_metrics["recall"][class_index]
            f1_delta = metrics["f1"][class_index] - clean_metrics["f1"][class_index]
            print(f"{condition:<12}{class_name:<25}"
                  f"{metrics['recall'][class_index]:>10.4f}"
                  f"{metrics['f1'][class_index]:>10.4f}"
                  f"{recall_delta:>+12.4f}{f1_delta:>+10.4f}")

fig, axes = plt.subplots(len(TEST_CONDITIONS), len(model_names), figsize=(14, 14))
for row, condition in enumerate(TEST_CONDITIONS):
    for column, model_name in enumerate(model_names):
        axis = axes[row, column]
        matrix = RESULTS[condition][model_name]["confusion_matrix"]
        image = axis.imshow(matrix, cmap="Blues")
        axis.set_title(f"{model_name} — {condition}")
        axis.set_xticks(range(NUM_CLASSES), labels=[name[0] for name in CLASS_NAMES])
        axis.set_yticks(range(NUM_CLASSES), labels=[name[0] for name in CLASS_NAMES])
        axis.set_xlabel("Predicted")
        axis.set_ylabel("True")
        for r in range(NUM_CLASSES):
            for c in range(NUM_CLASSES):
                axis.text(c, r, str(matrix[r, c]), ha="center", va="center",
                          color="white" if matrix[r, c] > matrix.max() / 2 else "black")
fig.tight_layout()
plt.show()

## Stage 8 — Interpret results and plan the next experiment

Use the metric table and per-class deltas above to discuss:

1. **Which classes are most affected?** Identify the largest negative change in per-class recall and F1, then inspect the corresponding confusion matrices.
2. **Is degradation gradual?** Compare each metric across Clean → 12 → 6 → 0 dB. A monotonic decline is gradual; a sharp final drop suggests a low-SNR failure region.
3. **Does preprocessing help?** Not answered by this experiment. `data_loader` has already filtered, segmented, normalized, and clipped each beat before this notebook adds noise. To measure preprocessing benefit, noise must be added to raw ECG records **before** the same preprocessing pipeline, and clean/noisy raw inputs must be evaluated using identical record splits and noise realizations.
4. **Could robustness improve?** As a controlled next experiment, train a second model with noise augmentation across the chosen SNR range, then compare it on this unchanged held-out noisy test set. Also investigate class-balanced/noise-aware augmentation and preprocessing choices; do not tune on the held-out test scores.

The FP32 NumPy path is expected to match the FP32 PyTorch path because both use the same linear weights and bias. A meaningful hardware edge comparison requires running the model on the target device or its supported runtime; the NumPy reference alone does not measure device-specific effects.